# Transformations and Adjustments

Before decomposing or modelling a time series it is often necessary to
**transform** the data so that patterns become clearer and models work better.
Transformations fall into two broad categories:

- **Adjustments** — correct for known external effects (calendar length,
  population growth, inflation) that distort the underlying signal.
- **Mathematical transformations** — stabilise variance, linearise trends, or
  make seasonal fluctuations roughly constant across the series.

This notebook covers the most common adjustments and transformations, with
particular attention to the **Box-Cox family** and practical guidance on when
and how to apply them.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# --- Alcohol Sales (monthly, 1992-2019) ---
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]

print("Airline shape:", airline.shape)
print("Alcohol shape:", alcohol.shape)
airline.head()

---
## 1. Calendar Adjustments

Monthly totals are misleading because months have different numbers of days.
January (31 days) will naturally have a higher total than February (28 days)
even if daily activity is identical.  The fix is simple: **divide by the number
of days in each month** to get a daily average.

$$
\bar{y}_t = \frac{y_t}{d_t}
$$

where $d_t$ is the number of days in month $t$.

In [ ]:
# Number of days in each month
days_in_month = airline.index.days_in_month

# Calendar-adjusted series: average passengers per day
airline["PassengersPerDay"] = airline["Passengers"] / days_in_month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline["Passengers"])
axes[0].set_title("Monthly Totals (raw)")
axes[0].set_ylabel("Thousands of Passengers")

axes[1].plot(airline["PassengersPerDay"], color="tab:orange")
axes[1].set_title("Average Passengers per Day (calendar-adjusted)")
axes[1].set_ylabel("Thousands per Day")

for ax in axes:
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

print("Calendar adjustment removes the artificial effect of differing month lengths.")

In [ ]:
# Look at February specifically — the shortest month
feb = airline[airline.index.month == 2][["Passengers", "PassengersPerDay"]]
feb.head(6)

---
## 2. Population Adjustments

If a quantity naturally grows with population (e.g., total electricity
consumption, total retail sales), computing the **per-capita** value isolates
genuine behavioural change from simple population growth:

$$
y_t^{\text{adj}} = \frac{y_t}{P_t}
$$

where $P_t$ is the population at time $t$.

We will simulate this with a synthetic example.

In [ ]:
# Synthetic example: total electricity consumption driven mostly by population growth
np.random.seed(42)
years = np.arange(2000, 2026)
population = 50_000 * (1.02 ** np.arange(len(years)))  # 2% annual growth
per_capita_usage = 10 + 0.5 * np.random.randn(len(years))  # roughly constant
total_usage = population * per_capita_usage

pop_df = pd.DataFrame({
    "Total": total_usage,
    "Population": population,
    "PerCapita": per_capita_usage,
}, index=pd.to_datetime(years, format="%Y"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pop_df.index, pop_df["Total"], marker="o", markersize=3)
axes[0].set_title("Total Electricity Consumption")
axes[0].set_ylabel("kWh")

axes[1].plot(pop_df.index, pop_df["PerCapita"], marker="o", markersize=3, color="tab:green")
axes[1].set_title("Per-Capita Consumption (population-adjusted)")
axes[1].set_ylabel("kWh per person")

plt.tight_layout()
plt.show()

print("The total shows a strong upward trend, but per-capita usage is essentially flat.")
print("Without the population adjustment we would falsely conclude demand is surging.")

---
## 3. Inflation Adjustments

Financial time series measured in current (nominal) dollars conflate real
changes in value with changes in the price level.  To isolate real changes,
**deflate** using a price index:

$$
y_t^{\text{real}} = \frac{y_t}{\text{CPI}_t / \text{CPI}_{\text{base}}}
$$

This converts all values to constant dollars relative to a chosen base period.

In [ ]:
# Synthetic example: nominal revenue with 3% annual inflation
np.random.seed(7)
months = pd.date_range("2010-01", periods=120, freq="MS")
cpi = 100 * (1.003 ** np.arange(len(months)))  # ~3% compounding monthly
real_revenue = 500 + 2 * np.sin(2 * np.pi * np.arange(len(months)) / 12) + np.random.randn(len(months)) * 5
nominal_revenue = real_revenue * (cpi / 100)

inf_df = pd.DataFrame({
    "Nominal": nominal_revenue,
    "CPI": cpi,
}, index=months)

# Deflate: convert to Jan-2010 dollars
cpi_base = inf_df["CPI"].iloc[0]
inf_df["Real"] = inf_df["Nominal"] / (inf_df["CPI"] / cpi_base)

fig, ax = plt.subplots()
ax.plot(inf_df["Nominal"], label="Nominal (current $)", alpha=0.8)
ax.plot(inf_df["Real"], label="Real (Jan-2010 $)", alpha=0.8)
ax.set_title("Nominal vs Real Revenue (inflation-adjusted)")
ax.set_ylabel("Revenue ($)")
ax.legend()
plt.tight_layout()
plt.show()

print("Nominal revenue trends upward, but real revenue is essentially flat.")

---
## 4. Mathematical Transformations

When seasonal variation **increases with the level** of the series (a hallmark
of multiplicative seasonality), a mathematical transformation can stabilise the
variance and make additive methods applicable.

### Common transforms

| Transform | Formula | When to use |
|---|---|---|
| Log | $w_t = \log(y_t)$ | Variance proportional to level; multiplicative effects |
| Square root | $w_t = \sqrt{y_t}$ | Variance proportional to $\sqrt{\text{level}}$; count data |
| Power | $w_t = y_t^p$ | General-purpose; $p < 1$ compresses large values |

In [ ]:
passengers = airline["Passengers"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
axes[0, 0].plot(passengers)
axes[0, 0].set_title("Original")
axes[0, 0].set_ylabel("Thousands")

# Log transform
axes[0, 1].plot(np.log(passengers), color="tab:orange")
axes[0, 1].set_title(r"Log transform: $w_t = \log(y_t)$")

# Square root
axes[1, 0].plot(np.sqrt(passengers), color="tab:green")
axes[1, 0].set_title(r"Square root: $w_t = \sqrt{y_t}$")

# Cube root (power p=1/3)
axes[1, 1].plot(passengers ** (1/3), color="tab:red")
axes[1, 1].set_title(r"Cube root: $w_t = y_t^{1/3}$")

for ax in axes.flat:
    ax.set_xlabel("Date")

plt.suptitle("Mathematical Transformations — Airline Passengers", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Notice how each transformation **compresses** the larger values more than the
smaller ones, making the seasonal swings more uniform across the series.
The log transform is the strongest; the square root is milder.

In [ ]:
# Demonstrate variance stabilisation quantitatively:
# split into first half and second half, compare seasonal amplitude
first_half = passengers[:72]
second_half = passengers[72:]

log_first = np.log(first_half)
log_second = np.log(second_half)

print("=== Original data ===")
print(f"  First half  std: {first_half.std():.2f}")
print(f"  Second half std: {second_half.std():.2f}")
print(f"  Ratio: {second_half.std() / first_half.std():.2f}")
print()
print("=== Log-transformed ===")
print(f"  First half  std: {log_first.std():.4f}")
print(f"  Second half std: {log_second.std():.4f}")
print(f"  Ratio: {log_second.std() / log_first.std():.2f}")
print()
print("A ratio near 1.0 means constant variance — the log transform achieves this.")

---
## 5. Box-Cox Transformation

The **Box-Cox transformation** generalises the log and power transforms into a
single family controlled by one parameter $\lambda$:

$$
w_t = \begin{cases}
\log(y_t) & \text{if } \lambda = 0 \\
\frac{y_t^\lambda - 1}{\lambda} & \text{otherwise}
\end{cases}
$$

Special cases:

| $\lambda$ | Transform |
|---|---|
| 1 | No transform (linear shift) |
| 0.5 | Square root (approximately) |
| 0 | Natural log |
| -0.5 | Reciprocal square root |
| -1 | Reciprocal |

The key insight: **the goal is to choose $\lambda$ so that seasonal variation
is roughly constant across the series**.

In [ ]:
# Visualise the effect of different lambda values
from scipy.stats import boxcox
from scipy.special import inv_boxcox

lambdas = [1.0, 0.5, 0.0, -0.5, -1.0]

fig, axes = plt.subplots(len(lambdas), 1, figsize=(12, 14), sharex=True)

for ax, lam in zip(axes, lambdas):
    if lam == 0:
        transformed = np.log(passengers.values)
    else:
        transformed = (passengers.values ** lam - 1) / lam
    ax.plot(passengers.index, transformed)
    ax.set_title(f"$\\lambda = {lam}$", fontsize=12)
    ax.set_ylabel("$w_t$")

axes[-1].set_xlabel("Date")
plt.suptitle("Box-Cox Transformation at Different $\\lambda$ Values", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Finding the optimal $\lambda$

`scipy.stats.boxcox` can automatically select the $\lambda$ that maximises the
log-likelihood of a normal distribution fitted to the transformed data.  In
practice this gives the $\lambda$ that best stabilises the variance.

In [ ]:
# Optimal lambda via scipy
transformed, optimal_lambda = boxcox(passengers.values)

print(f"Optimal lambda: {optimal_lambda:.4f}")
print(f"\nInterpretation: lambda ~ {optimal_lambda:.2f} is close to 0 (log transform),")
print("confirming that a log transform is appropriate for airline passengers.")

In [ ]:
# Visualise: original vs Box-Cox transformed with optimal lambda
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(passengers)
axes[0].set_title("Original — Airline Passengers")
axes[0].set_ylabel("Thousands")

axes[1].plot(passengers.index, transformed, color="tab:purple")
axes[1].set_title(f"Box-Cox Transformed ($\\lambda$ = {optimal_lambda:.4f})")
axes[1].set_ylabel("$w_t$")

for ax in axes:
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

print("The seasonal amplitude is now roughly constant across the entire series.")

In [ ]:
# Apply Box-Cox to Alcohol Sales for a second example
sales = alcohol["Sales"].dropna()

sales_transformed, sales_lambda = boxcox(sales.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sales)
axes[0].set_title("Alcohol Sales — Original")
axes[0].set_ylabel("Millions of Dollars")

axes[1].plot(sales.index, sales_transformed, color="tab:orange")
axes[1].set_title(f"Box-Cox Transformed ($\\lambda$ = {sales_lambda:.4f})")
axes[1].set_ylabel("$w_t$")

for ax in axes:
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

print(f"Optimal lambda for Alcohol Sales: {sales_lambda:.4f}")

### Box-Cox confidence interval for $\lambda$

`scipy.stats.boxcox_normmax` and `boxcox` provide point estimates.  We can
also examine the log-likelihood profile to see how sensitive the result is to
the exact choice of $\lambda$.

In [ ]:
# Log-likelihood profile for lambda
from scipy.stats import boxcox_llf

lam_range = np.linspace(-1, 2, 200)
llf_values = [boxcox_llf(lam, passengers.values) for lam in lam_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lam_range, llf_values)
ax.axvline(optimal_lambda, color="tab:red", linestyle="--",
           label=f"Optimal $\\lambda$ = {optimal_lambda:.4f}")
ax.set_title("Box-Cox Log-Likelihood Profile — Airline Passengers")
ax.set_xlabel("$\\lambda$")
ax.set_ylabel("Log-Likelihood")
ax.legend()
plt.tight_layout()
plt.show()

print("The curve is fairly flat near the peak, so nearby lambda values")
print("(e.g., lambda=0 i.e. log) would work almost as well.")

---
## 6. Back-Transformation

After modelling or forecasting on transformed data, we must **reverse** the
transform to get predictions back on the original scale.

The inverse Box-Cox is:

$$
y_t = \begin{cases}
e^{w_t} & \text{if } \lambda = 0 \\
(\lambda w_t + 1)^{1/\lambda} & \text{otherwise}
\end{cases}
$$

`scipy.special.inv_boxcox` handles this automatically.

In [ ]:
# Forward transform
transformed_vals, lam = boxcox(passengers.values)

# Back-transform
recovered = inv_boxcox(transformed_vals, lam)

# Verify round-trip
max_error = np.max(np.abs(recovered - passengers.values))
print(f"Maximum round-trip error: {max_error:.2e}")
print("Effectively zero — the back-transform perfectly recovers the original data.")

In [ ]:
# Visual confirmation of round-trip
fig, ax = plt.subplots()
ax.plot(passengers.index, passengers.values, label="Original", linewidth=2)
ax.plot(passengers.index, recovered, label="Back-transformed", linestyle="--", linewidth=2)
ax.set_title("Round-Trip: Original vs Back-Transformed")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate back-transform for simple log case
log_vals = np.log(passengers.values)
back_from_log = np.exp(log_vals)

print("Log back-transform (exp):")
print(f"  Max error: {np.max(np.abs(back_from_log - passengers.values)):.2e}")
print()
print("Tip: when using log-transformed data for forecasting, remember that")
print("exp(mean of log-forecasts) != mean of forecasts on original scale.")
print("A bias adjustment may be needed (see Jensen's inequality).")

---
## 7. When to Transform — Decision Guide

Not every series needs a transformation. Here is a practical decision
framework:

1. **Plot the data.** Look for seasonal amplitude that changes with the level.
2. **Check rolling standard deviation.** If it trends with the level, a
   transform is warranted.
3. **Try a log transform first.** It is the most common, easy to interpret
   (percentage changes), and easy to back-transform.
4. **Use Box-Cox if log is not enough.** Let `scipy.stats.boxcox` choose
   $\lambda$, but prefer a "nice" value ($0$, $0.5$, $1$) if the
   log-likelihood profile is flat enough.
5. **Do not transform if:**
   - The series has zero or negative values (Box-Cox requires positive data).
   - The seasonal variation is already constant.
   - Interpretability on the original scale is critical and a simpler model
     suffices.

In [ ]:
# Diagnostic: rolling std vs rolling mean scatter plot
# If correlated => transform; if flat => no need
rolling_mean = passengers.rolling(12).mean()
rolling_std = passengers.rolling(12).std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before transform
axes[0].scatter(rolling_mean, rolling_std, alpha=0.6, edgecolors="k", linewidths=0.5)
axes[0].set_title("Rolling Std vs Rolling Mean — Original")
axes[0].set_xlabel("Rolling Mean")
axes[0].set_ylabel("Rolling Std")

# After log transform
log_passengers = np.log(passengers)
log_rolling_mean = log_passengers.rolling(12).mean()
log_rolling_std = log_passengers.rolling(12).std()

axes[1].scatter(log_rolling_mean, log_rolling_std, alpha=0.6, edgecolors="k",
                linewidths=0.5, color="tab:orange")
axes[1].set_title("Rolling Std vs Rolling Mean — Log Transformed")
axes[1].set_xlabel("Rolling Mean")
axes[1].set_ylabel("Rolling Std")

plt.tight_layout()
plt.show()

print("Left: clear positive relationship — variance grows with level (transform needed).")
print("Right: no relationship — variance is stable (transform succeeded).")

---
## Summary

| Adjustment / Transform | Purpose | When to apply |
|---|---|---|
| Calendar adjustment | Remove month-length artefacts | Monthly totals |
| Population adjustment | Isolate per-capita change | Data that scales with population |
| Inflation adjustment | Convert to constant dollars | Financial data over long horizons |
| Log transform | Stabilise variance; interpret as % change | Multiplicative seasonality |
| Square root | Milder variance stabilisation | Count data |
| Box-Cox ($\lambda$) | General variance stabilisation | When log alone is not ideal |

**Key principle:** transformations should be applied *before* decomposition or
modelling, and forecasts must be *back-transformed* to the original scale
before being reported.

In [ ]:
# Clean up helper columns
airline.drop(columns=["PassengersPerDay"], inplace=True, errors="ignore")